In [2]:
import pandas as pd
import numpy as np
import timeit


def load_and_clean_power_data(file_path):
    df = pd.read_csv(file_path, sep=';', 
                     low_memory=False, 
                     na_values=['?'])
    
    df = df.dropna()
    
    numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    df[numeric_cols] = df[numeric_cols].astype(float)

    df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
    
    return df

In [3]:

df_power = load_and_clean_power_data('household_power_consumption.txt')

print(df_power.head())
print(f"Total rows: {len(df_power)}")

         Date      Time  Global_active_power  Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00                4.216                  0.418   234.84   
1  16/12/2006  17:25:00                5.360                  0.436   233.63   
2  16/12/2006  17:26:00                5.374                  0.498   233.29   
3  16/12/2006  17:27:00                5.388                  0.502   233.74   
4  16/12/2006  17:28:00                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  \
0              18.4             0.0             1.0            17.0   
1              23.0             0.0             1.0            16.0   
2              23.0             0.0             2.0            17.0   
3              23.0             0.0             1.0            17.0   
4              15.8             0.0             1.0            17.0   

             DateTime  
0 2006-12-16 17:24:00  
1 2006-12-16 17:25:00  
2 2006-12-16 17:26:0

In [4]:
def filter_high_power(df):
    return df[df['Global_active_power'] > 5]


def filter_intensity_and_appliances(df):
    mask = (df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20)
    result = df[mask & (df['Sub_metering_2'] > df['Sub_metering_3'])]
    return result

def random_sample_means(df):
    sample = df.sample(n=500000, replace=False)
    means = sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return means

def complex_filter(df):
    time_mask = (df['DateTime'].dt.hour >= 18) & (df['Global_active_power'] > 6)
    initial_subset = df[time_mask]
    
    final_subset = initial_subset[
        (initial_subset['Sub_metering_2'] > initial_subset['Sub_metering_1']) & 
        (initial_subset['Sub_metering_2'] > initial_subset['Sub_metering_3'])
    ]
    
    mid = len(final_subset) // 2
    first_half = final_subset.iloc[:mid:3] 
    second_half = final_subset.iloc[mid::4]  
    
    return pd.concat([first_half, second_half])

In [5]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

def process_statistics(df):
    scaler_minmax = MinMaxScaler()
    df_norm = pd.DataFrame(scaler_minmax.fit_transform(df[['Global_active_power', 'Voltage']]), 
                           columns=['Power_Norm', 'Voltage_Norm'])
    
    scaler_std = StandardScaler()
    df_std = pd.DataFrame(scaler_std.fit_transform(df[['Global_active_power', 'Voltage']]), 
                          columns=['Power_Std', 'Voltage_Std'])

    pearson = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')
    spearman = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')
    
    print(f"Pearson: {pearson:.4f}, Spearman: {spearman:.4f}")

    df['Day_Type'] = np.where(df['DateTime'].dt.dayofweek < 5, 'Weekday', 'Weekend')
    df_encoded = pd.get_dummies(df, columns=['Day_Type'])
    
    return df_encoded.head()

In [8]:
t1 = timeit.timeit(lambda: filter_high_power(df_power), number=1)
t2 = timeit.timeit(lambda: filter_intensity_and_appliances(df_power), number=1)
t3 = timeit.timeit(lambda: random_sample_means(df_power), number=1)
t4 = timeit.timeit(lambda: complex_filter(df_power), number=1)

print(f"Task 1 (Power > 5kW): {t1:.6f} s")
print(f"Task 2 (Intensity 19-20A): {t2:.6f} s")
print(f"Task 3 (Random 500k): {t3:.6f} s")
print(f"Task 4 (Complex filter): {t4:.6f} s")

Task 1 (Power > 5kW): 0.006800 s
Task 2 (Intensity 19-20A): 0.013900 s
Task 3 (Random 500k): 0.202800 s
Task 4 (Complex filter): 0.048800 s


In [9]:
final_df_head = process_statistics(df_power)
print("\nOne Hot Encoded Data (Head):")
print(final_df_head[['DateTime', 'Day_Type_Weekday', 'Day_Type_Weekend']].head())

Pearson: 0.9989, Spearman: 0.9954

One Hot Encoded Data (Head):
             DateTime  Day_Type_Weekday  Day_Type_Weekend
0 2006-12-16 17:24:00             False              True
1 2006-12-16 17:25:00             False              True
2 2006-12-16 17:26:00             False              True
3 2006-12-16 17:27:00             False              True
4 2006-12-16 17:28:00             False              True
